# Signal Sandbox

Prototype a signal idea end-to-end: define it, backtest it, see the result.

**Workflow:** Edit the signal cell, then Run All below it.

In [ ]:
from _setup import *

## 1. Load data

Change `SYMBOL` to test on a different asset, or set `SYMBOL = None` to load the full universe.

In [ ]:
SYMBOL = "ETH-USD"    # single asset, or None for full universe
START  = "2017-01-01"
END    = "2026-12-31"
COST_BPS = 20.0       # round-trip transaction cost

panel = load_daily_bars(start=START, end=END)

if SYMBOL:
    asset = panel[panel["symbol"] == SYMBOL].copy().sort_values("ts").set_index("ts")
    close = asset["close"]
    high = asset["high"]
    low = asset["low"]
    volume = asset["volume"]
    returns = close.pct_change(fill_method=None).dropna()
    print(f"Loaded {SYMBOL}: {len(close)} bars, {close.index.min()} to {close.index.max()}")
else:
    panel = filter_universe(panel, min_adv_usd=500_000, min_history_days=90)
    close_wide = panel.pivot(index="ts", columns="symbol", values="close")
    volume_wide = panel.pivot(index="ts", columns="symbol", values="volume")
    returns_wide = close_wide.pct_change(fill_method=None)
    universe_wide = panel.pivot(index="ts", columns="symbol", values="in_universe").fillna(False).astype(bool)
    print(f"Loaded universe: {len(close_wide)} days, {close_wide.columns.size} symbols")

---
## 2. Define signal

**Edit this cell.** The signal should be a pandas Series with the same index as `close`.

- `1.0` = long
- `0.0` = flat (cash)
- Fractional values (e.g. 0.5) are fine for partial sizing.
- For universe-wide signals, produce a DataFrame with same shape as `close_wide`.

In [ ]:
# ── EDIT THIS ────────────────────────────────────────────────────────
#
# Example: SMA(5, 40) crossover — long when fast > slow
#
FAST = 5
SLOW = 40

ma_fast = close.rolling(FAST).mean()
ma_slow = close.rolling(SLOW).mean()
signal = (ma_fast > ma_slow).astype(float)
#
# ─────────────────────────────────────────────────────────────────────

## 3. Signal diagnostics

In [ ]:
sig = signal.dropna()
tim = float(sig.mean())
n_trades = int((sig.diff().abs() > 0.5).sum())

print(f"Signal length:    {len(sig)} bars")
print(f"Time in market:   {tim:.1%}")
print(f"Number of trades: {n_trades}")
print(f"Avg trade length: {len(sig) * tim / max(n_trades, 1):.0f} bars")

## 4. Backtest

In [ ]:
# Build weights — for single-asset, wrap signal into a 1-column DataFrame
if SYMBOL:
    weights = signal.to_frame(name=SYMBOL)
    ret_df = returns.to_frame(name=SYMBOL)
else:
    weights = signal  # should already be a DataFrame
    ret_df = returns_wide

# Run backtest
strat = quick_backtest(weights, ret_df, cost_bps=COST_BPS, label="Signal")

# Buy-and-hold benchmark
if SYMBOL:
    bh_weights = pd.DataFrame(1.0, index=weights.index, columns=[SYMBOL])
else:
    bh_weights = universe_wide.astype(float).div(universe_wide.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

bh = quick_backtest(bh_weights, ret_df, cost_bps=COST_BPS, label="Buy & Hold")

results = [strat, bh]

## 5. Metrics

In [ ]:
metrics_table(results)

## 6. Equity curves

In [ ]:
plot_equity(results, title=f"{SYMBOL or 'Universe'} — Equity Curves")
plt.show()

## 7. Drawdowns

In [ ]:
plot_drawdowns(results, title=f"{SYMBOL or 'Universe'} — Drawdowns")
plt.show()

## 8. Signal + price overlay

In [ ]:
if SYMBOL:
    fig, ax1 = plt.subplots(figsize=(14, 5))

    ax1.plot(close.index, close.values, color=NAVY, lw=0.8, label="Close")
    ax1.set_ylabel("Price", color=NAVY)
    ax1.set_title(f"{SYMBOL} — Signal overlay", fontweight="bold")

    ax2 = ax1.twinx()
    ax2.fill_between(sig.index, sig.values, alpha=0.15, color=TEAL, step="post")
    ax2.set_ylabel("Signal (0=cash, 1=long)", color=TEAL)
    ax2.set_ylim(-0.1, 1.5)

    ax1.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("Signal overlay is only available for single-asset mode (set SYMBOL).")